# Train the soccer-ball detector on Colab (T4, 16GB)

Your 6GB laptop GPU capped training at imgsz 960. A T4 has 16GB, so we can train at **1280** (what the ~15px ball needs) with a bigger model and all the frames.

**Split:** train here → download `best.pt` → put it in your local `models/ball.pt` → run live on your own GPU.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Your data is at `My Drive/newTrackingData/train/SNMOT-###` (set `SNMOT_DIR` below if it moves).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- CONFIG: Drive folder that CONTAINS the SNMOT-* sequences (folders or zips) ----
SNMOT_DIR = '/content/drive/MyDrive/newTrackingData/train'

MODEL   = 'yolov8s.pt'   # live-friendly; try 'yolov8m.pt' for max accuracy (slower locally)
IMGSZ   = 1280           # the whole point of using the T4
EPOCHS  = 60
BATCH   = 12             # T4 fits yolov8s @1280 ~batch 12; drop to 8 if it OOMs
OUT_WEIGHTS = '/content/drive/MyDrive/soccer_ball_best.pt'   # where to save the result

In [ ]:
# ---- Stage the SNMOT sequences on LOCAL disk (training over the Drive mount is far too slow) ----
import glob, zipfile, os, shutil
os.makedirs('/content/snmot', exist_ok=True)
zips    = sorted(glob.glob(os.path.join(SNMOT_DIR, 'SNMOT-*.zip')))
folders = [f for f in sorted(glob.glob(os.path.join(SNMOT_DIR, 'SNMOT-*'))) if os.path.isdir(f)]
if zips:
    for z in zips:
        zipfile.ZipFile(z).extractall('/content/snmot')
elif folders:
    for src in folders:                       # copy extracted folders off Drive to local disk
        dst = os.path.join('/content/snmot', os.path.basename(src))
        if not os.path.exists(dst):
            shutil.copytree(src, dst)
else:
    raise AssertionError(f'no SNMOT zips or folders found in {SNMOT_DIR}')
seqs = sorted(glob.glob('/content/snmot/SNMOT-*'))
print(f'{len(seqs)} sequences staged locally:', [os.path.basename(s) for s in seqs])

In [ ]:
# ---- Build the YOLO ball dataset from the ground truth ----
import re
VAL_SEQUENCES = {'SNMOT-099', 'SNMOT-100'}   # held out for validation (edit if you don't have these)
TRAIN_STRIDE  = 1   # 16GB + T4 can handle every frame; use 2-3 for a quicker run

def ball_track_id(gameinfo):
    for line in open(gameinfo):
        m = re.match(r'\s*trackletID_(\d+)\s*=\s*ball', line, re.I)
        if m:
            return int(m.group(1))

def seq_dims(seqinfo):
    w = h = None
    for line in open(seqinfo):
        if line.startswith('imWidth'):  w = int(line.split('=')[1])
        if line.startswith('imHeight'): h = int(line.split('=')[1])
    return w, h

def process(seq):
    bid = ball_track_id(os.path.join(seq, 'gameinfo.ini'))
    W, H = seq_dims(os.path.join(seq, 'seqinfo.ini'))
    ball = {}
    for line in open(os.path.join(seq, 'gt', 'gt.txt')):
        p = line.strip().split(',')
        if len(p) >= 6 and int(p[1]) == bid:
            ball[int(p[0])] = (float(p[2]), float(p[3]), float(p[4]), float(p[5]))
    imgs = []
    for img in sorted(glob.glob(os.path.join(seq, 'img1', '*.jpg'))):
        fr = int(os.path.splitext(os.path.basename(img))[0])
        lab = os.path.splitext(img)[0] + '.txt'
        if fr in ball:
            x, y, bw, bh = ball[fr]
            with open(lab, 'w') as f:
                f.write(f'0 {(x+bw/2)/W:.6f} {(y+bh/2)/H:.6f} {bw/W:.6f} {bh/H:.6f}\n')
        else:
            open(lab, 'w').close()
        imgs.append(img)
    return imgs

train, val = [], []
for seq in seqs:
    imgs = process(seq)
    if os.path.basename(seq) in VAL_SEQUENCES:
        val += imgs[::2]
    else:
        train += imgs[::TRAIN_STRIDE]
assert val, 'no val images -- edit VAL_SEQUENCES to two sequence names you actually have'

os.makedirs('/content/datasets/ball', exist_ok=True)
open('/content/datasets/ball/train.txt', 'w').write('\n'.join(train) + '\n')
open('/content/datasets/ball/val.txt',   'w').write('\n'.join(val) + '\n')
with open('/content/datasets/ball/dataset.yaml', 'w') as f:
    f.write('train: /content/datasets/ball/train.txt\n')
    f.write('val: /content/datasets/ball/val.txt\n')
    f.write('nc: 1\nnames: [ball]\n')
print(f'train {len(train)} | val {len(val)}')

In [ ]:
# ---- Train ----
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data='/content/datasets/ball/dataset.yaml',
    imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=0,
    patience=20, project='runs', name='ball', exist_ok=True,
    mosaic=1.0, close_mosaic=10, scale=0.5, translate=0.1,
)

In [ ]:
# ---- Validate with the metric that matters: ball found near its true position ----
# (mAP50 is misleadingly harsh on a 15px box; this is 'within 25px of ground truth'.)
import cv2, numpy as np
best = YOLO('runs/detect/ball/weights/best.pt')

def gt_centers(seq):
    bid = ball_track_id(os.path.join(seq, 'gameinfo.ini')); d = {}
    for line in open(os.path.join(seq, 'gt', 'gt.txt')):
        p = line.strip().split(',')
        if len(p) >= 6 and int(p[1]) == bid:
            d[int(p[0])] = (float(p[2])+float(p[4])/2, float(p[3])+float(p[5])/2)
    return d

hit = tot = 0
for seq in [s for s in seqs if os.path.basename(s) in VAL_SEQUENCES]:
    gt = gt_centers(seq)
    for img in sorted(glob.glob(os.path.join(seq, 'img1', '*.jpg')))[::5]:
        fr = int(os.path.splitext(os.path.basename(img))[0])
        if fr not in gt: continue
        tot += 1
        r = best(cv2.imread(img), imgsz=IMGSZ, conf=0.15, device=0, verbose=False)[0]
        gx, gy = gt[fr]
        for b in r.boxes:
            x1, y1, x2, y2 = map(float, b.xyxy[0])
            if np.hypot((x1+x2)/2-gx, (y1+y2)/2-gy) < 25:
                hit += 1; break
print(f'ball within 25px of GT: {hit}/{tot} = {100*hit/tot:.0f}%   (local yolov8n@960 was 34%)')

In [ ]:
# ---- Save the weights back to Drive, then download them ----
import shutil
shutil.copy('runs/detect/ball/weights/best.pt', OUT_WEIGHTS)
print('saved to', OUT_WEIGHTS)
from google.colab import files
files.download('runs/detect/ball/weights/best.pt')

## Back on your laptop

1. Put the downloaded `best.pt` at `models/ball.pt` (overwrite the old one).
2. If you trained at 1280, bump `BALL_MODEL_IMGSZ` in `src/config.py` (e.g. 1280 for accuracy, or keep 960 for live speed — the model still works at either).
3. Re-run `python run.py --source data/match.mp4` — the pipeline auto-picks up the new `models/ball.pt`.

If yolov8s is too slow in your live loop, retrain with `MODEL='yolov8n.pt'` — you still get the 1280-resolution benefit while keeping nano's speed.